# Pytroch Titanic

In [1]:
import sys
import os
import torch

print(os.getcwd())
sys.path.append('../ml/clean-code')

from day4_titanic_cleaning import load_data, clean_data, encoding
from titanic_logistic_regression import train_test_split_data



c:\Users\Edward\daily-ds\deep-learning


In [2]:
from sklearn.preprocessing import StandardScaler


df = load_data('../ml/data/train.csv')
df = clean_data(df)
df = encoding(df)
X_train, X_test, y_train, y_test = train_test_split_data(df, 0.2)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)
print(X_train_tensor.shape)
print(y_train_tensor.shape)
#print(X_test_tensor)
#print(y_test_tensor)

torch.Size([711, 9])
torch.Size([711, 1])


In [3]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle =True)
for X_batch, y_batch in train_loader:
    print(X_batch.shape, y_batch.shape)


torch.Size([32, 9]) torch.Size([32, 1])
torch.Size([32, 9]) torch.Size([32, 1])
torch.Size([32, 9]) torch.Size([32, 1])
torch.Size([32, 9]) torch.Size([32, 1])
torch.Size([32, 9]) torch.Size([32, 1])
torch.Size([32, 9]) torch.Size([32, 1])
torch.Size([32, 9]) torch.Size([32, 1])
torch.Size([32, 9]) torch.Size([32, 1])
torch.Size([32, 9]) torch.Size([32, 1])
torch.Size([32, 9]) torch.Size([32, 1])
torch.Size([32, 9]) torch.Size([32, 1])
torch.Size([32, 9]) torch.Size([32, 1])
torch.Size([32, 9]) torch.Size([32, 1])
torch.Size([32, 9]) torch.Size([32, 1])
torch.Size([32, 9]) torch.Size([32, 1])
torch.Size([32, 9]) torch.Size([32, 1])
torch.Size([32, 9]) torch.Size([32, 1])
torch.Size([32, 9]) torch.Size([32, 1])
torch.Size([32, 9]) torch.Size([32, 1])
torch.Size([32, 9]) torch.Size([32, 1])
torch.Size([32, 9]) torch.Size([32, 1])
torch.Size([32, 9]) torch.Size([32, 1])
torch.Size([7, 9]) torch.Size([7, 1])


In [4]:
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(in_features=9, out_features=16),
    nn.Sigmoid(),
    nn.Linear(in_features=16, out_features=8),
    nn.Sigmoid(),
    nn.Linear(in_features=8, out_features=1),
    nn.Sigmoid()
)

loss_fn = torch.nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [5]:
for epoch in range(20):
    for X_batch, y_batch in train_loader:
        prediction = model(X_batch)
        loss = loss_fn(prediction, y_batch)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 0.6858764290809631
Epoch 1, Loss: 0.6586466431617737
Epoch 2, Loss: 0.5669957399368286
Epoch 3, Loss: 0.4477148950099945
Epoch 4, Loss: 0.5908729434013367
Epoch 5, Loss: 0.3671046793460846
Epoch 6, Loss: 0.5604555606842041
Epoch 7, Loss: 0.34923672676086426
Epoch 8, Loss: 0.3567672669887543
Epoch 9, Loss: 0.3784959316253662
Epoch 10, Loss: 0.7377534508705139
Epoch 11, Loss: 0.581089437007904
Epoch 12, Loss: 0.4507865011692047
Epoch 13, Loss: 0.9337202310562134
Epoch 14, Loss: 0.8196545243263245
Epoch 15, Loss: 0.23739643394947052
Epoch 16, Loss: 0.3119988739490509
Epoch 17, Loss: 0.3452070653438568
Epoch 18, Loss: 0.12262246757745743
Epoch 19, Loss: 0.3104493319988251


# Evaluation

In [6]:
model.eval()
with torch.no_grad():
    train_predictions = model(X_train_tensor)
    test_predictions = model(X_test_tensor)

print(train_predictions.shape)
print(train_predictions[:10])

train_pred_classes = (train_predictions >= 0.5).float()
test_pred_classes = (test_predictions >= 0.5).float()

train_accuracy = (train_pred_classes == y_train_tensor).float().mean()
test_accuracy = (test_pred_classes == y_test_tensor).float().mean()

print(f"Train accuracy: {train_accuracy.item()}")
print(f"Test accuracy: {test_accuracy.item()}")


torch.Size([711, 1])
tensor([[0.3286],
        [0.1492],
        [0.3692],
        [0.9640],
        [0.3934],
        [0.1183],
        [0.9602],
        [0.3804],
        [0.7015],
        [0.3929]])
Train accuracy: 0.8227847814559937
Test accuracy: 0.8314606547355652
